In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

# DOANH THU, SỐ CHƯƠNG TRÌNH KHUYẾN MÃI VÀ TỔNG LƯỢNG ĐƠN HÀNG QUA CÁC MỐC THỜI GIAN

# Lợi nhuận thu được

In [ ]:
dfSale =  pd.read_csv("Analytical/sales.csv")
dfSale.info()

In [ ]:
dfSale["Date"] = pd.to_datetime(dfSale["Date"])
dfSale["Profit"] = dfSale["Revenue"] - dfSale["COGS"]
dfSale["yearMonth"] = dfSale["Date"].dt.strftime("%Y-%m")
dfSale["year"] = dfSale["Date"].dt.year
dfSale["month"] = dfSale["Date"].dt.month
dfSale.info()

In [ ]:
profitbyYear = dfSale.groupby("year")["Profit"].sum().reset_index().sort_values(by = "year")
profitbyYear

In [ ]:
# profitbyYear = dfSale.groupby("year")["Profit"].sum()
# year = dfSale[]
pig, ax =  plt.subplots(figsize=(15, 5))
ax.bar(
    profitbyYear['year'],
    profitbyYear['Profit']
)
ax.set_ylabel('Lợi nhuận)')
ax.set_xlabel('Năm', fontsize=12)
ax.grid(ls = "--")
ax.set_xticks(range(min(profitbyYear['year']),max(profitbyYear['year'])+1))
plt.show()

In [ ]:
dfSales = dfSale.groupby(["year","month"])["Profit"].sum().reset_index()
pd.pivot_table(dfSales, values='Profit', index=['year'], columns=['month'], aggfunc='sum').fillna(0)

In [ ]:
profitbyyearMonth = dfSale.groupby("yearMonth")["Revenue"].sum()
pig, ax =  plt.subplots(figsize=(20, 9))
ax.bar(
    profitbyyearMonth.index,
    profitbyyearMonth.values
)
ax.set_ylabel('Lợi nhuận)')
ax.set_xlabel('Tháng', fontsize=12)
plt.xticks(rotation=270)
ax.grid(ls = "--")

plt.show()

In [ ]:
# Lượng đơn hàng
dfOrder_item = pd.read_csv("Transaction/order_items.csv")
dfOrder = pd.read_csv("Transaction/orders.csv")
dfOrder_item.info()

In [ ]:
# Số lượng đơn hàng
dfOrder["order_date"] = pd.to_datetime(dfOrder["order_date"])
dfOrder["yearMonth"] = dfOrder["order_date"].dt.strftime("%Y-%m")
dfOrder["year"] = dfOrder["order_date"].dt.year
dfOrder["month"] = dfOrder["order_date"].dt.month


In [ ]:
countOrder = dfOrder.groupby("yearMonth")["order_id"].count()

pig, ax =  plt.subplots(figsize=(20, 9))
ax.plot(
    countOrder.index,
    countOrder.values,
    marker = "o"
)
ax.set_ylabel('Lợi nhuận)')
ax.set_xlabel('Tháng', fontsize=12)
plt.xticks(rotation=270)
ax.grid(ls = "--")

plt.show()

## Chương trình khuyến mại

In [ ]:
dfPromotions = pd.read_csv("Master/promotions.csv")
dfPromotions.info()

In [ ]:
dfPromotions["start_date"] = pd.to_datetime(dfPromotions["start_date"])
dfPromotions["end_date"] = pd.to_datetime(dfPromotions["end_date"])
dfPromotions['year'] = dfPromotions['start_date'].dt.year
dfPromotions['month'] = dfPromotions['start_date'].dt.month
dfPromotions['duration'] = (dfPromotions['end_date'] - dfPromotions['start_date']).dt.days
dfPromotions["yearMonth"] = dfPromotions["start_date"].dt.strftime("%Y-%m")


In [ ]:
dfPromotions.head()

In [ ]:
dfOrderFull = pd.merge(dfOrder, dfOrder_item, on='order_id', how='inner')

In [ ]:
dfOrderFull.head()

In [ ]:
dfProduct = pd.read_csv("Master/products.csv")

In [ ]:
# Số tiền để dành cho khuyến mãi
dfDisCountAmount_Year =  dfOrderFull.groupby(["year"])["discount_amount"].sum().reset_index().sort_values(by = ['year'])
pig, ax =  plt.subplots(figsize=(15, 5))
ax.bar(
    dfDisCountAmount_Year["year"],
    dfDisCountAmount_Year["discount_amount"],
    
)
ax.set_title("BIỂU ĐỒ CỘT SỐ TIỀN DÀNH ĐỂ GIẢM GIÁ QUA CÁC NĂM")
ax.set_ylabel('TIỀN')
ax.set_xlabel('NĂM', fontsize=12)
ax.grid(ls = "--")
ax.set_xticks(range(2012,2023))
plt.show()

In [ ]:
dfDisCountAmount_year_Month =  dfOrderFull.groupby(["year",'month'])["discount_amount"].sum().reset_index().sort_values(by = ['year', 'month'])
dfDisCountAmount_year_Month['date'] = pd.to_datetime(dfDisCountAmount_year_Month[['year', 'month']].assign(DAY=1))
dfDisCountAmount_year_Month = dfDisCountAmount_year_Month.sort_values('date')


x0 = dfDisCountAmount_year_Month['date'].map(pd.Timestamp.toordinal)
y0 = dfDisCountAmount_year_Month['discount_amount']


coeffs0 = np.polyfit(x0, y0, 1)
trend0 = np.poly1d(coeffs0)
trend_values0 = trend0(x0)


plt.figure(figsize=(14, 6))


# plt.subplot(3,2,1)
plt.plot(
    dfDisCountAmount_year_Month['date'],
    dfDisCountAmount_year_Month['discount_amount'],
    label='discount_amount',
    marker='o',
    color='#DD8452')


plt.plot(
    dfDisCountAmount_year_Month['date'],
    trend_values0,
    linestyle='--',
    linewidth=2,
    label=f'Trend slope: {coeffs0[0]:.6f}'
)


plt.title('SỐ TIỀN CHI TIÊU CHO CÁC VOURCHE GIẢM GIÁ(2012 - 2022)')
plt.xlabel('Thời gian')
plt.ylabel('Số lượng')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)


plt.show()

In [ ]:
dfDisCountAmount_year_Month =  dfOrderFull.groupby(["year",'month'])["discount_amount"].sum().reset_index().sort_values(by = ['year', 'month'])

In [ ]:
dfDisCountAmount_year_Month.sort_values(by = "discount_amount", ascending = False)[:20]

In [ ]:
min_year = dfDisCountAmount_year_Month["year"].min()
max_year = dfDisCountAmount_year_Month["year"].max()

sizeOfYear = (dfDisCountAmount_year_Month["year"].unique())

numCol =  2
numRow =  int(round(len(sizeOfYear)/2,0))
indexPlot = 1
plt.figure(figsize=(20,15))
for year in range(min_year, max_year + 1):
    plt.subplot(numRow, numCol, indexPlot)
    
    df_year = dfDisCountAmount_year_Month[
        dfDisCountAmount_year_Month["year"] == year
    ]
    
    plt.plot(
        df_year["month"],
        df_year["discount_amount"],
        label=f"{year}",
        marker = 'o'
    )
    plt.legend()
    plt.xticks(range(1,13))  
    plt.grid(ls = "--")
    indexPlot += 1
plt.show()

# Quản lý kho

In [ ]:
dfWareHouse  = pd.read_csv("Operational/inventory.csv")
df_plot = dfWareHouse.copy()
dfWareHouse.info()

In [ ]:
dfWareHouse.describe()

In [ ]:
monthly_trend = dfWareHouse.groupby(['year', 'month'])[['units_sold', 'units_received']].sum().reset_index()
monthly_trend.info()

## Tổng quan về kho

In [ ]:
monthly_trend['per'] = (monthly_trend['units_sold']/monthly_trend['units_received'])

In [ ]:
monthly_trend['date'] = pd.to_datetime(monthly_trend[['year', 'month']].assign(DAY=1))
monthly_trend = monthly_trend.sort_values('date')

# Tùy chỉnh Matplotlib line chart
plt.figure(figsize=(14, 6))
plt.plot(monthly_trend['date'], monthly_trend['units_sold'], label='Units Sold', marker='o', color='#DD8452')
plt.plot(monthly_trend['date'], monthly_trend['units_received'], label='Units Received', marker='s', color='#4C72B0', alpha=0.7)

plt.title('Xu hướng Nhập - Xuất hàng theo thời gian (2012 - 2022)')
plt.xlabel('Thời gian')
plt.ylabel('Số lượng')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
x = monthly_trend['date'].map(pd.Timestamp.toordinal)
y1 = monthly_trend['units_sold']


coeffs1 = np.polyfit(x, y1, 1)
trend1 = np.poly1d(coeffs1)
trend_values1 = trend1(x)


plt.figure(figsize=(14, 10))


plt.subplot(3,2,1)
plt.plot(monthly_trend['date'], monthly_trend['units_sold'], label='units_sold', marker='o', color='#DD8452')
plt.plot(
    monthly_trend['date'],
    trend_values1,
    linestyle='--',
    linewidth=2,
    label=f'Trend slope: {coeffs1[0]:.6f}'
)


plt.title('HÀNG NHẬP(2012 - 2022)')
plt.xlabel('Thời gian')
plt.ylabel('Số lượng')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)



plt.subplot(3,2,2)
y2 = monthly_trend['units_received']


coeffs2 = np.polyfit(x, y2, 1)
trend2 = np.poly1d(coeffs2)
trend_values2 = trend2(x)

plt.plot(monthly_trend['date'], monthly_trend['units_sold'], label='units_sold', marker='o', color='#DD8452')
plt.plot(
    monthly_trend['date'],
    trend_values2,
    linestyle='--',
    linewidth=2,
    label=f'Trend slope: {coeffs2[0]:.6f}'
)
plt.title('HÀNG BÁN (2012 - 2022)')
plt.xlabel('Thời gian')
plt.ylabel('Số lượng')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)


plt.subplot(3,2,3)
y3 = monthly_trend['per']


coeffs3 = np.polyfit(x, y3, 1)
trend3 = np.poly1d(coeffs3)
trend_values3 = trend3(x)

plt.plot(monthly_trend['date'], monthly_trend['per'], label='units_sold', marker='o', color='#DD8452')
plt.plot(
    monthly_trend['date'],
    trend_values3,
    linestyle='--',
    linewidth=2,
    label=f'Trend slope: {coeffs3[0]:.6f}'
)

plt.title('TỈ LỆ TIÊU THỤ GIỮA HÀNG NHẬP - BÁN (2012 - 2022)')
plt.xlabel('Thời gian')
plt.ylabel('Tỉ lệ')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)


plt.tight_layout()
plt.show()

In [ ]:
monthly_trend
pd.pivot_table(monthly_trend, values='per', index=['year'], columns=['month'], aggfunc='sum').fillna(0)

In [ ]:
import seaborn as sns
plt.figure(figsize=(10, 8))

# Lọc các cột số (numeric) cần phân tích
corr_cols = ['stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'days_of_supply', 'fill_rate', 'sell_through_rate','stockout_flag', 'overstock_flag']
corr = dfWareHouse[corr_cols].corr()

# Tùy chỉnh Seaborn heatmap
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Biểu đồ Heatmap Tương quan (Correlation) các chỉ số Inventory')
plt.tight_layout()
plt.show()

**Phân tích này giúp xác định cách các chỉ số chuỗi cung ứng tác động qua lại lẫn nhau:**
- Tỷ lệ Bán ra (Sell-through rate): Chỉ số này có tương quan âm (-0.37) với stock_on_hand và days_of_supply. Nghĩa là khi lượng hàng tồn hoặc số ngày lưu kho tăng lên, tốc độ bán ra thực tế lại có xu hướng giảm. Đây là một điểm nghẽn (bottleneck) đáng chú ý, cho thấy dấu hiệu đọng vốn do overstock và là một chủ đề rất tốt để đưa ra thảo luận trong các buổi brainstorming sắp tới cùng team.

## Tình trạng thiếu hàng

In [ ]:
# Tình hình thiếu hàng ở từng danh mục sản phẩm
dfstockout_flag = dfWareHouse.groupby(["year","category"])["stockout_flag"].mean().reset_index().sort_values(by = 'year', ascending=False)
pd.pivot_table(dfstockout_flag, values='stockout_flag', index=['category'], columns=['year'], aggfunc='sum')

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 6))

fig.suptitle('Tần suất xảy ra tình trạng hết hàng', fontsize=13)

ax[0,0].plot(dfstockout_flag['year'].unique(),
        dfstockout_flag[dfstockout_flag['category'] == "Streetwear"]["stockout_flag"],
        color='#2196F3',
        linewidth=2.5,
        marker='o',
        markersize=7,
        label='Streetwear')
ax[0,0].set_xticks(range(min(dfstockout_flag['year']),max(dfstockout_flag['year'])+1))
ax[0,0].grid(ls = "--")
ax[0,0].set_title("Theo Streetwear")
ax[0,0].legend()


# Bieu do 2
# plt.subplot(2,2,2)
ax[0,1].plot(dfstockout_flag['year'].unique(),
        dfstockout_flag[dfstockout_flag['category'] == "GenZ"]["stockout_flag"],
        color='#4CAF50',
        linewidth=2.5,
        marker='s',
        markersize=7,
        label='GenZ')
ax[0,1].set_xticks(range(min(dfstockout_flag['year']),max(dfstockout_flag['year'])+1))
ax[0,1].grid(ls = "--")
ax[0,1].set_title("Theo GenZ")
ax[0,1].legend()


# Bieu do 1
# plt.subplot(2,2,3)
ax[1,0].plot(dfstockout_flag['year'].unique(),
        dfstockout_flag[dfstockout_flag['category'] == "Casual"]["stockout_flag"],
        color='#FF5722',
        linewidth=2.5,
        marker='^',
        markersize=8,
        label='Casual')
ax[1,0].set_xticks(range(min(dfstockout_flag['year']),max(dfstockout_flag['year'])+1))
ax[1,0].grid(ls = "--")
ax[1,0].set_title("Theo Casual")
ax[1,0].legend()


# Bieu do 1
# plt.subplot(2,2,4)
ax[1,1].plot(dfstockout_flag['year'].unique(),
        dfstockout_flag[dfstockout_flag['category'] == "Outdoor"]["stockout_flag"],
        color='darkorange',
        linewidth=2.5,
        marker='*',
        markersize=8,
        label='Outdoor')
ax[1,1].set_xticks(range(min(dfstockout_flag['year']),max(dfstockout_flag['year'])+1))
ax[1,1].grid(ls = "--")
ax[1,1].set_title("Theo Outdoor")
ax[1,1].legend()


plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = (18, 4)
pd.pivot_table(dfstockout_flag, values='stockout_flag', index=['year'], columns=['category'], aggfunc='sum').plot(kind = 'bar', stacked = False)
plt.xticks(rotation = 0)
plt.grid(ls = "--")
plt.legend(title='Category', loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.show()

In [ ]:
# Số ngày hết hàng trong tháng của mỗi sản phẩm
dfstockout_days = dfWareHouse.groupby(["year","category"])["stockout_days"].mean().sort_values(ascending=False).reset_index().sort_values(by = 'year', ascending=False)
pd.pivot_table(dfstockout_days, values='stockout_days', index=['category'], columns=['year'], aggfunc='sum')

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 6))

fig.suptitle('Số ngày hết hàng trung bình theo năm và danh mục', fontsize=13)

ax[0,0].plot(dfstockout_days['year'].unique(),
        dfstockout_days[dfstockout_days['category'] == "Streetwear"]["stockout_days"],
        color='#2196F3',
        linewidth=2.5,
        marker='o',
        markersize=7,
        label='Streetwear')
ax[0,0].set_xticks(range(min(dfstockout_days['year']),max(dfstockout_days['year'])+1))
ax[0,0].grid(ls = "--")
ax[0,0].set_title("Theo Streetwear")
ax[0,0].legend()


# Bieu do 2
# plt.subplot(2,2,2)
ax[0,1].plot(dfstockout_days['year'].unique(),
        dfstockout_days[dfstockout_days['category'] == "GenZ"]["stockout_days"],
        color='#4CAF50',
        linewidth=2.5,
        marker='s',
        markersize=7,
        label='GenZ')
ax[0,1].set_xticks(range(min(dfstockout_days['year']),max(dfstockout_days['year'])+1))
ax[0,1].grid(ls = "--")
ax[0,1].set_title("Theo GenZ")
ax[0,1].legend()


# Bieu do 1
# plt.subplot(2,2,3)
ax[1,0].plot(dfstockout_days['year'].unique(),
        dfstockout_days[dfstockout_days['category'] == "Casual"]["stockout_days"],
        color='#FF5722',
        linewidth=2.5,
        marker='^',
        markersize=8,
        label='Casual')
ax[1,0].set_xticks(range(min(dfstockout_days['year']),max(dfstockout_days['year'])+1))
ax[1,0].grid(ls = "--")
ax[1,0].set_title("Theo Casual")
ax[1,0].legend()


# Bieu do 1
# plt.subplot(2,2,4)
ax[1,1].plot(dfstockout_days['year'].unique(),
        dfstockout_days[dfstockout_days['category'] == "Outdoor"]["stockout_days"],
        color='darkorange',
        linewidth=2.5,
        marker='*',
        markersize=8,
        label='Outdoor')
ax[1,1].set_xticks(range(min(dfstockout_days['year']),max(dfstockout_days['year'])+1))
ax[1,1].grid(ls = "--")
ax[1,1].set_title("Theo Outdoor")
ax[1,1].legend()


plt.tight_layout()
plt.show()

## Thừa hàng

In [ ]:
# Hàng hóa gặp tình trạng vượt quá mức cần thiết
dfMeanoverstock_flag = dfWareHouse.groupby(["year","category"])["overstock_flag"].mean().reset_index().sort_values(by = 'year', ascending=False)
pd.pivot_table(dfMeanoverstock_flag, values='overstock_flag', index=['category'], columns=['year'], aggfunc='sum')

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 6))

fig.suptitle('Tần suất xảy ra tình trạng thừa hàng theo các năm', fontsize=13)

ax[0,0].plot(dfMeanoverstock_flag['year'].unique(),
        dfMeanoverstock_flag[dfMeanoverstock_flag['category'] == "Streetwear"]["overstock_flag"],
        color='#2196F3',
        linewidth=2.5,
        marker='o',
        markersize=7,
        label='Streetwear')
ax[0,0].set_xticks(range(min(dfMeanoverstock_flag['year']),max(dfMeanoverstock_flag['year'])+1))
ax[0,0].grid(ls = "--")
ax[0,0].set_title("Theo Streetwear")
ax[0,0].legend()


# Bieu do 2
# plt.subplot(2,2,2)
ax[0,1].plot(dfMeanoverstock_flag['year'].unique(),
        dfMeanoverstock_flag[dfMeanoverstock_flag['category'] == "GenZ"]["overstock_flag"],
        color='#4CAF50',
        linewidth=2.5,
        marker='s',
        markersize=7,
        label='GenZ')
ax[0,1].set_xticks(range(min(dfMeanoverstock_flag['year']),max(dfMeanoverstock_flag['year'])+1))
ax[0,1].grid(ls = "--")
ax[0,1].set_title("Theo GenZ")
ax[0,1].legend()


# Bieu do 1
# plt.subplot(2,2,3)
ax[1,0].plot(dfMeanoverstock_flag['year'].unique(),
        dfMeanoverstock_flag[dfMeanoverstock_flag['category'] == "Casual"]["overstock_flag"],
        color='#FF5722',
        linewidth=2.5,
        marker='^',
        markersize=8,
        label='Casual')
ax[1,0].set_xticks(range(min(dfMeanoverstock_flag['year']),max(dfMeanoverstock_flag['year'])+1))
ax[1,0].grid(ls = "--")
ax[1,0].set_title("Theo Casual")
ax[1,0].legend()


# Bieu do 1
# plt.subplot(2,2,4)
ax[1,1].plot(dfMeanoverstock_flag['year'].unique(),
        dfMeanoverstock_flag[dfMeanoverstock_flag['category'] == "Outdoor"]["overstock_flag"],
        color='darkorange',
        linewidth=2.5,
        marker='*',
        markersize=8,
        label='Outdoor')
ax[1,1].set_xticks(range(min(dfMeanoverstock_flag['year']),max(dfMeanoverstock_flag['year'])+1))
ax[1,1].grid(ls = "--")
ax[1,1].set_title("Theo Outdoor")
ax[1,1].legend()


plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = (18, 4)
pd.pivot_table(dfMeanoverstock_flag, values='overstock_flag', index=['year'], columns=['category'], aggfunc='sum').plot(kind = 'bar', stacked = False)
plt.xticks(rotation = 0)
plt.grid(ls = "--")
plt.legend(title='Category', loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.show()

In [ ]:
# Loại hàng cần tái đặt hàng sớm nhiều nhất
dfCategory_Reorder_Flag = dfWareHouse.groupby(["year","category"])["reorder_flag"].sum().sort_values(ascending=False).reset_index()
pd.pivot_table(dfCategory_Reorder_Flag, values='reorder_flag', index=['category'], columns=['year'], aggfunc='sum')

# SỐ LƯỢNG ĐƠN HÀNG + TRẢ HÀNG

In [ ]:
dfOrder_item = pd.read_csv("Transaction/order_items.csv")
dfOrder = pd.read_csv("Transaction/orders.csv")
dfOrder_item.info()

In [ ]:
dfOrder.info()

In [ ]:
# Số lượng đơn hàng
dfOrder["order_date"] = pd.to_datetime(dfOrder["order_date"])
dfOrder["yearMonth"] = dfOrder["order_date"].dt.strftime("%Y-%m")
dfOrder["year"] = dfOrder["order_date"].dt.year
dfOrder["month"] = dfOrder["order_date"].dt.month
dfOrder.head()

In [ ]:
dfDisCountOrderID =  dfOrderFull.groupby(["year",'month'])["order_id"].count().reset_index().sort_values(by = ['year', 'month'])
dfDisCountOrderID['date'] = pd.to_datetime(dfDisCountOrderID[['year', 'month']].assign(DAY=1))
dfDisCountOrderID = dfDisCountOrderID.sort_values('date')


x_count_orderID = dfDisCountOrderID['date'].map(pd.Timestamp.toordinal)
y_count_orderID = dfDisCountOrderID['order_id']


coeffs_count_orderID = np.polyfit(x_count_orderID, y_count_orderID, 1)
trend_count_orderID = np.poly1d(coeffs_count_orderID)
trend_values_count_orderID = trend_count_orderID(x_count_orderID)


plt.figure(figsize=(14, 6))


# plt.subplot(3,2,1)
plt.plot(
    dfDisCountOrderID['date'],
    dfDisCountOrderID['order_id'],
    label='order_id',
    marker='o',
    color='#DD8452')


plt.plot(
    dfDisCountOrderID['date'],
    trend_values_count_orderID,
    linestyle='--',
    linewidth=2,
    label=f'Trend slope: {coeffs_count_orderID[0]:.6f}'
)


plt.title('SỐ LƯỢNG ĐƠN HÀNG(2012 - 2022)')
plt.xlabel('Thời gian')
plt.ylabel('Số lượng')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)


plt.show()

In [ ]:
dfPayment = pd.read_csv("Transaction/payments.csv")
df = pd.merge(dfPayment, dfOrder, on='order_id', how='inner')
# Gom dữ liệu theo từng năm
years = sorted(df["year"].unique())
data = [df[df["year"] == y]["payment_value"] for y in years]

plt.figure(figsize=(10,6))

plt.boxplot(data, labels=years,showmeans=True)

plt.title("Phân bổ giá trị đơn hàng theo năm")
plt.xlabel("NĂM")
plt.ylabel("GIÁ TRỊ MỖI ĐƠN HÀNG")

plt.show()

1. Xu hướng chung: giá trị đơn hàng đang tăng theo thời gian
Median (đường màu cam trong box) tăng đều từ 2012 → 2022
Điều này cho thấy:
→ Khách hàng đang chi tiêu nhiều hơn qua từng năm
→ Hoặc giá sản phẩm / chiến lược pricing đã thay đổi
2. Độ phân tán ngày càng lớn (variance tăng)
Box (IQR) và whisker ngày càng cao hơn theo năm
Nghĩa là:
→ Hành vi khách hàng không còn đồng đều
→ Có sự phân hóa rõ:
Nhóm mua ít
Nhóm mua rất nhiều

👉 Đây là dấu hiệu của:

mở rộng tệp khách hàng
hoặc thêm nhiều phân khúc sản phẩm (giá rẻ → cao cấp)

3. Outliers cực nhiều và tăng mạnh
Các chấm đen phía trên rất dày, đặc biệt từ 2017 trở đi
Có những đơn hàng rất lớn (200k–300k+)

→ Điều này thường là:

đơn bulk / B2B
hoặc campaign lớn / khách VIP

👉 Nhưng cũng cần cảnh giác:

Có thể là data skew → mean sẽ bị kéo lệch rất mạnh

4. Median tăng nhưng không quá đột biến
Tăng đều, không có năm nào “nhảy vọt” quá lớn
→ Tăng trưởng khá ổn định, không phải kiểu spike bất thường
5. Insight quan trọng (điểm ăn tiền nếu đưa vào portfolio)
Doanh thu tăng có thể không đến từ toàn bộ khách hàng
→ mà đến từ một nhóm nhỏ khách chi tiêu lớn (outliers)
Nếu bạn chỉ nhìn mean → sẽ bị “ảo tưởng tăng trưởng”
Boxplot giúp thấy:
→ phần lớn khách vẫn chi tiêu ở mức trung bình vừa phải

In [ ]:
dfReturn  = pd.read_csv("Transaction/returns.csv")
dfReturn.info()

In [ ]:
dfReturn[dfReturn["order_id"] == 698128]

In [ ]:
# dfReturn["return_date"] = pd.to_datetime(dfReturn["return_date"])
# dfReturn["year"] = dfReturn["return_date"].dt.year
# dfReturn["month"] = dfReturn["return_date"].dt.month
dfReturn.head()

In [ ]:
dfReturn_mer_dfOrder = pd.merge(dfReturn, dfOrder, on= ["order_id"], how = "inner")
dfReturn_mer_dfOrder

In [ ]:
totalOrderReturnByYear = dfReturn_mer_dfOrder.groupby(["year",'month'])["order_id"].nunique().reset_index().sort_values(by = ['year', 'month']).rename(columns={"order_id": "count_order_return"})
totalOrderByYear = dfOrder.groupby(["year",'month'])["order_id"].nunique().reset_index().sort_values(by = ['year', 'month']).rename(columns={"order_id": "count_order"})

In [ ]:
perOrderReturnByYear =  pd.merge(totalOrderReturnByYear, totalOrderByYear, on= ["year", "month"], how = "inner")

In [ ]:
perOrderReturnByYear = perOrderReturnByYear.groupby(["year"])[["count_order_return","count_order"]].sum().reset_index()

In [ ]:
perOrderReturnByYear['per'] = perOrderReturnByYear['count_order_return']/perOrderReturnByYear['count_order']
perOrderReturnByYear

In [ ]:
# perOrderReturnByYear.groupby('year')['per'].mean().reset_index()

In [ ]:
# perOrderReturnByYear['date'] = pd.to_datetime(perOrderReturnByYear[['year', 'month']].assign(DAY=1))
perOrderReturnByYear = perOrderReturnByYear.sort_values('year')

# perOrderReturnByYear = perOrderReturnByYear.groupby(["year"])[["per"]].mean().reset_index()

# x_countOrderReturn = perOrderReturnByYear['date'].map(pd.Timestamp.toordinal)
x_countOrderReturn = perOrderReturnByYear['year']
y_countOrderReturn = perOrderReturnByYear['per']

coeffs_countOrderReturn = np.polyfit(x_countOrderReturn, y_countOrderReturn, 1)
trend_countOrderReturn = np.poly1d(coeffs_countOrderReturn)
trend_values_countOrderReturn = trend_countOrderReturn(x_countOrderReturn)


plt.figure(figsize=(14, 6))


# plt.subplot(3,2,1)
plt.plot(
    perOrderReturnByYear['year'],
    perOrderReturnByYear['per'],
    label='Số lượng đơn hàng được trả lại',
    marker='o',
    color='#DD8452')


plt.plot(
    perOrderReturnByYear['year'],
    trend_values_countOrderReturn,
    linestyle='--',
    linewidth=2,
    label=f'Trend slope: {coeffs_countOrderReturn[0]:.6f}'
)


plt.title('SỐ LƯỢNG ĐƠN HÀNG DÍNH BỊ TRẢ HÀNG(2012 - 2022)')
plt.xlabel('Thời gian')
plt.ylabel('Số lượng')
plt.xticks(range(min(perOrderReturnByYear['year']),max(perOrderReturnByYear['year'])+1))  
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# totalItemInOrder = dfOrderFull.groupby(["year","month","order_id"])["product_id"].count().reset_index().rename(columns={"product_id": "count_item_order"})
# totalItemReturnInOrder = dfReturn.groupby(["order_id"])["product_id"].count().reset_index().rename(columns={"product_id": "count_item_return"})
# perItemReturnInOrder = pd.merge(totalItemInOrder, totalItemReturnInOrder, on = 'order_id', how = 'left').fillna(0)

In [ ]:
# perItemReturnInOrder['per'] = perItemReturnInOrder["count_item_return"] / perItemReturnInOrder["count_item_order"]*100

In [ ]:
# perItemReturnInOrder = perItemReturnInOrder.groupby(["year","month"])["per"].mean().reset_index()


In [ ]:
# perItemReturnInOrder['date'] = pd.to_datetime(perItemReturnInOrder[['year', 'month']].assign(DAY=1))
# perItemReturnInOrder = perItemReturnInOrder.sort_values('date')


# x_perItemReturnInOrder = perItemReturnInOrder['date'].map(pd.Timestamp.toordinal)
# y_perItemReturnInOrder = perItemReturnInOrder['per']


# coeffs_perItemReturnInOrder = np.polyfit(x_perItemReturnInOrder, y_perItemReturnInOrder, 1)
# trend_perItemReturnInOrder = np.poly1d(coeffs_perItemReturnInOrder)
# trend_values_perItemReturnInOrder = trend_perItemReturnInOrder(x_perItemReturnInOrder)


# plt.figure(figsize=(14, 6))


# # plt.subplot(3,2,1)
# plt.plot(
#     perItemReturnInOrder['date'],
#     perItemReturnInOrder['per'],
#     label='Số lượng đơn hàng được trả lại',
#     marker='o',
#     color='#DD8452')


# plt.plot(
#     perItemReturnInOrder['date'],
#     trend_values_perItemReturnInOrder,
#     linestyle='--',
#     linewidth=2,
#     label=f'Trend slope: {coeffs_perItemReturnInOrder[0]:.6f}'
# )


# plt.title('SỐ LƯỢNG ĐƠN HÀNG(2012 - 2022)')
# plt.xlabel('Thời gian')
# plt.ylabel('Số lượng')
# plt.legend()
# plt.grid(True, linestyle='--', alpha=0.5)
# plt.show()

**Nhận xét**

- Tính mùa vụ cực cao sẵn sàng bán lỗ trong 1 khoảng thời gian

# SẢN PHẨM + KHUYẾN MẠI + ĐƠN HÀNG + TRẢ HÀNG + NHẬN XÉT

In [ ]:
dfProduct = pd.read_csv("Master/products.csv")

In [ ]:
dfProduct.sort_values(by = 'product_id')

## KHÁCH HÀNG VÀ VỊ TRÍ ĐỊA LÝ